# 09 · Knowledge Graphs & RotatE

## What this notebook covers
Knowledge graphs (KGs) represent facts as triples (head, relation, tail) — e.g., (London, capital_of, UK). KG embedding models learn vector representations of entities and relations to enable *link prediction*: inferring missing triples.

## TransE — the foundational model (narrative)
TransE (Bordes et al. 2013) was the first successful KG embedding model. The idea is beautifully simple: model each relation as a translation vector r such that h + r ≈ t for valid triples. Minimise ||h + r - t|| for positive triples; maximise it for negatives.

TransE works well for 1-to-1 relations but fails on:
- **Symmetric relations**: (A, spouse_of, B) implies (B, spouse_of, A) — but h + r ≠ t + r in general
- **Hierarchical 1-to-N**: (Paris, located_in, France) and (Lyon, located_in, France) can't both satisfy h + r = t

**RotatE** (Sun et al. 2019) fixed this by modelling relations as rotations in complex space. Each entity is a complex vector; each relation rotates the head entity onto the tail. This elegantly handles symmetry, anti-symmetry, inversion, and composition patterns.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
torch.manual_seed(42); np.random.seed(42)

# Synthetic KG: 20 entities, 5 relation types, ~200 triples
n_ent, n_rel = 20, 5
np.random.seed(0)
all_triples = set()
# Generate structured triples per relation
for r in range(n_rel):
    entities = np.random.choice(n_ent, 8, replace=False)
    for i in range(len(entities) - 1):
        all_triples.add((int(entities[i]), r, int(entities[i+1])))
        if r == 0:  # symmetric relation
            all_triples.add((int(entities[i+1]), r, int(entities[i])))

triples = np.array(list(all_triples))
print(f"KG: {n_ent} entities  {n_rel} relations  {len(triples)} triples")

# Train/test split
np.random.shuffle(triples)
n_train = int(len(triples) * 0.8)
train_triples = triples[:n_train]
test_triples  = triples[n_train:]

In [ ]:
class RotatE(nn.Module):
    """
    RotatE: entities as complex vectors, relations as rotations.
    Score(h,r,t) = -||h ∘ r - t||  where ∘ is element-wise complex multiplication.
    """
    def __init__(self, n_ent, n_rel, embed_dim=64, gamma=12.0):
        super().__init__()
        self.gamma = gamma
        self.embed_dim = embed_dim // 2  # half for real, half for imaginary
        self.ent_emb = nn.Embedding(n_ent, embed_dim)
        self.rel_emb = nn.Embedding(n_rel, embed_dim // 2)  # angles only
        nn.init.uniform_(self.ent_emb.weight, -1/embed_dim**0.5, 1/embed_dim**0.5)
        nn.init.uniform_(self.rel_emb.weight, -np.pi, np.pi)

    def score(self, h_idx, r_idx, t_idx):
        h = self.ent_emb(h_idx)
        t = self.ent_emb(t_idx)
        theta = self.rel_emb(r_idx)  # rotation angles

        h_re, h_im = h[:, :self.embed_dim], h[:, self.embed_dim:]
        t_re, t_im = t[:, :self.embed_dim], t[:, self.embed_dim:]
        r_re = torch.cos(theta)
        r_im = torch.sin(theta)

        # Complex rotation: (h_re + i*h_im) * (r_re + i*r_im)
        hr_re = h_re * r_re - h_im * r_im
        hr_im = h_re * r_im + h_im * r_re

        diff_re = hr_re - t_re
        diff_im = hr_im - t_im
        dist = torch.sqrt(diff_re**2 + diff_im**2 + 1e-8).sum(dim=-1)
        return self.gamma - dist

def negative_sample(triples, n_ent, n_neg=1):
    negs = []
    for h, r, t in triples:
        for _ in range(n_neg):
            if np.random.rand() < 0.5:
                negs.append((np.random.randint(n_ent), r, t))
            else:
                negs.append((h, r, np.random.randint(n_ent)))
    return np.array(negs)

model = RotatE(n_ent, n_rel, embed_dim=64)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

losses = []
for epoch in range(200):
    negs = negative_sample(train_triples, n_ent)
    pos_t = torch.LongTensor(train_triples)
    neg_t = torch.LongTensor(negs)
    pos_score = model.score(pos_t[:,0], pos_t[:,1], pos_t[:,2])
    neg_score = model.score(neg_t[:,0], neg_t[:,1], neg_t[:,2])
    loss = F.softplus(-pos_score).mean() + F.softplus(neg_score).mean()
    opt.zero_grad(); loss.backward(); opt.step()
    losses.append(loss.item())

print(f"Final training loss: {losses[-1]:.4f}")

In [ ]:
# Evaluate: MRR and Hits@k on test triples
@torch.no_grad()
def evaluate_kge(model, test_triples, n_ent, k_vals=(1, 3, 10)):
    mrr_sum = 0
    hits = {k: 0 for k in k_vals}
    for h, r, t in test_triples:
        # Score h against all entities as tail
        h_t  = torch.LongTensor([h] * n_ent)
        r_t  = torch.LongTensor([r] * n_ent)
        all_t = torch.arange(n_ent)
        scores = model.score(h_t, r_t, all_t).numpy()
        rank = (-scores).argsort().tolist().index(t) + 1
        mrr_sum += 1.0 / rank
        for k in k_vals:
            if rank <= k:
                hits[k] += 1
    n = len(test_triples)
    return mrr_sum/n, {k: hits[k]/n for k in k_vals}

mrr, hits = evaluate_kge(model, test_triples, n_ent)
print(f"RotatE on synthetic KG:")
print(f"  MRR: {mrr:.4f}")
for k, v in hits.items():
    print(f"  Hits@{k}: {v:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(losses, color="steelblue"); axes[0].set_title("RotatE training loss"); axes[0].set_xlabel("Epoch")

# Entity embedding 2D vis
from sklearn.manifold import TSNE
ent_emb = model.ent_emb.weight.detach().numpy()
ent_2d = TSNE(n_components=2, random_state=0, perplexity=5).fit_transform(ent_emb)
axes[1].scatter(ent_2d[:,0], ent_2d[:,1], c=np.arange(n_ent), cmap="tab20", s=80)
for i in range(n_ent): axes[1].annotate(str(i), ent_2d[i], fontsize=7)
axes[1].set_title("Entity embeddings (t-SNE)")

plt.tight_layout()
plt.savefig(f"{base}/09_rotate_kg.png", dpi=120)
plt.show()